# Extract Type Library from DKG Ontology
Make sure you have all packages installed before running the code.

In [1]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase, RoutingControl, Result
import pandas as pd

## Connect to ontology db

In [2]:
# load secrets from .env file
load_dotenv()
onto_reader = os.getenv("ONTO_READER")
onto_reader_password = os.getenv("ONTO_READER_PASSWORD")
onto_url = os.getenv("ONTO_URL")

In [3]:
# Define and check connection of DKG
url = onto_url
user = onto_reader # your username
password = onto_reader_password # your password
db_name = "neo4j" # or your db name
driver = GraphDatabase.driver(url, auth=(user, password))

driver.verify_connectivity()

## Extract metadata sources

In [4]:
with driver.session() as session:
    query = (
            '''
            match (n:MetadataSource)
            RETURN n;
            '''
            )
    result = session.run(query)
    records = [record['n'] for record in result]
    metadata_sources = pd.DataFrame(records)    

## Extract asset areas

In [5]:
with driver.session() as session:
    query = (
            '''
            match (n:AssetArea)
            RETURN n;
            '''
            )
    result = session.run(query)
    records = [(record['n']) for record in result]
    asset_areas = pd.DataFrame(records)    

In [6]:
with driver.session() as session:
    query = (
            '''
            match (n:AssetArea)-[]-(m:MetadataSource)
            RETURN n.name as aaName, m.name as msName;
            '''
            )
    result = session.run(query)
    records = [(record['aaName'], record['msName']) for record in result]
    aa_ms = pd.DataFrame(records, columns=['name', 'metadata_source_name'])    

## Extract asset area types

In [7]:
with driver.session() as session:
    query = (
            '''
            match (n:AssetAreaType)
            RETURN n;
            '''
            )
    result = session.run(query)
    records = [record['n'] for record in result]
    asset_area_types = pd.DataFrame(records)    

## Extract relationship types between metadata asset types

In [8]:
with driver.session() as session:
    query = (
            '''
            match (n:AssetType)-[r]->(m:AssetType)
            where n.status='tested' and m.status='tested'
            return r;
            '''
    )
    result = session.run(query)
    records = [record['r'] for record in result]
    relation_types_tested = pd.DataFrame(records)

In [9]:
with driver.session() as session:
    query = (
            '''
            match (n:AssetType)-[r]->(m:AssetType)
            where n.status='tested' and m.status='tested'
            return r.id as RelationId, n.name as sAssetType, m.name as tAssetType;
            '''
    )
    result = session.run(query)
    records = [(record['RelationId'], record['sAssetType'], record['tAssetType']) for record in result]
    source_rel_target = pd.DataFrame(records, columns=['id', 'source_asset_type_name', 'target_asset_type_name'])

## Extract tested asset types

In [10]:
with driver.session() as session:
    query = (
            '''
            match (n:AssetType)
            where n.status='tested'
            return n;
            '''
            )
    result = session.run(query)
    records = [record['n'] for record in result]
    asset_types_tested = pd.DataFrame(records)

In [18]:
with driver.session() as session:
    query = (
            '''
            match (n:AssetType{status:'tested'})-[r]->(aa:AssetArea)<-[]-(ms:MetadataSource)
            RETURN n.name as name, aa.id as aaID, aa.name as aaName, r.note as atNote, ms.id as msID, ms.name as msName;
            '''
            )
    result = session.run(query)
    records = [(record['name'],record['aaID'],  record['aaName'], 
                record['atNote'], record['msID'], record['msName']) for record in result]
    at_aa_ms = pd.DataFrame(records, columns=['name', 'asset_area_id', 'asset_area_name', 
                                              'asset_note', 'metadata_source_id', 'metadata_source_name'])    

## Extract the attribute types of the tested asset types

In [ ]:
with driver.session() as session:
    query = (
            '''
            match (n:AssetType)-[r]->(m:AttributeType)
            where n.status='tested'
            return distinct m;
            '''
            )
    result = session.run(query)
    records = [record['m'] for record in result]
    attribute_types_tested = pd.DataFrame(records)

## Extract status

In [ ]:
with driver.session() as session:
    query = (
            '''
            match (n:Status)
            return n;
            '''
            )
    result = session.run(query)
    records = [record['n'] for record in result]
    status = pd.DataFrame(records)

## Extract schemas

In [ ]:
def extract_schema(label):
    with driver.session() as session:
        query = (
                f'MATCH (n:{label}) RETURN n;'
        )
        result = session.run(query)
        records = [record["n"] for record in result]
        return pd.DataFrame([dict(records[0])])

In [15]:
asset_schema = extract_schema('AssetSchema')
attribute_schema = extract_schema('AttributeSchema')
asset_area_schema = extract_schema('AssetAreaSchema')
relation_schema = extract_schema('RelationSchema')
metadata_source_schema = extract_schema('MetadataSourceSchema')

Output the dataframes to your preferred file format at your preferred location for easy use.